In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import Tensor
from torch.distributions import Normal
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, build_target, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)

## CONVERGENCE, large network (deep wide), boston split 0, 4 chains from 4 MAPs

Each sticky sampler was run 4 times on the same posterior, chain $c$ started from its own MAP (Adam from its own random init, seed $90000 + c$), with the reference covariance of MAP 0 shared by all chains. $K = 10^6$ events, minibatch 128, $w = 0.05$, $S = 4000$ draws per chain.

**Step 1.** Load the MAPs and the chains, and see how different the starting points are. Raw parameter distances say little, since permuting hidden units or flipping signs gives a different vector with the same network. So the MAPs are also compared in function space, through their predictions.

In [ ]:
import itertools
import sys
sys.path.insert(0, "notebooks")
from utils.uci_results_utils import predict

CONV_DIR = Path("results/paper_v2/deep_wide_convergence/deep_wide/boston/split_00")
CONV_SAMPLERS = {"sticky_zigzag": "Sticky ZigZag", "sticky_boomerang": "Sticky Boomerang"}
N_CONV_CHAINS = 4

X_all, y_all = load_raw_datasets(("boston",))["boston"]
cdata = make_split(X_all, y_all, seed=BASE_SEED + 0, dtype=torch.float64, device="cpu")
c_ystd = cdata["y_std"]


def conv_rmse(f: Tensor, y: Tensor) -> float:
    """RMSE of the predictions f against y, in original units."""
    return float(((f - y) ** 2).mean().sqrt()) * c_ystd


# --- the 4 MAPs ---------------------------------------------------------------
maps = [torch.load(CONV_DIR / "maps" / f"map_{i:02d}.pt", map_location="cpu", weights_only=False)
        for i in range(N_CONV_CHAINS)]
LAYERS = maps[0]["layer_sizes"]
x_maps = torch.stack([m["x_ref"].double() for m in maps])               # [4, D]
f_map_test = predict(x_maps[:, :-1], LAYERS, "tanh", cdata["X_test"])    # [4, N_test]
f_map_train = predict(x_maps[:, :-1], LAYERS, "tanh", cdata["X_train"])  # [4, N_train]

print(f"network {LAYERS}, D = {x_maps.shape[1]}\n")
print(f"{'MAP':<5}{'seed':>7}{'energy':>9}{'train RMSE':>12}{'test RMSE':>11}{'sigma':>8}{'|beta|':>8}")
for i, m in enumerate(maps):
    print(f"{i:<5}{m['map_seed']:>7}{m['energy']:>9.1f}"
          f"{conv_rmse(f_map_train[i], cdata['y_train']):>12.3f}{conv_rmse(f_map_test[i], cdata['y_test']):>11.3f}"
          f"{float(x_maps[i, -1].exp()) * c_ystd:>8.3f}{float(x_maps[i, :-1].norm()):>8.2f}")

print("\npairwise differences")
print(f"{'pair':<6}{'param (relative)':>18}{'test predictions (RMS, y units)':>34}")
for i, j in itertools.combinations(range(N_CONV_CHAINS), 2):
    rel = float((x_maps[i, :-1] - x_maps[j, :-1]).norm()
                / (0.5 * (x_maps[i, :-1].norm() + x_maps[j, :-1].norm())))
    print(f"{i}-{j:<4}{rel:>18.2f}{conv_rmse(f_map_test[i], f_map_test[j]):>34.3f}")

# --- the 8 chains (memory-mapped, nothing is read until used) ------------------
def load_chain(c: int, sampler: str) -> dict:
    for name in (sampler, f"grid_{sampler}"):
        p = CONV_DIR / f"chain_{c}" / f"{name}.pt"
        if p.exists():
            return torch.load(p, map_location="cpu", weights_only=False, mmap=True)
    raise FileNotFoundError(f"chain {c} {sampler}")

conv = {s: [load_chain(c, s) for c in range(N_CONV_CHAINS)] for s in CONV_SAMPLERS}

print("\nchains")
for s, label in CONV_SAMPLERS.items():
    for c, r in enumerate(conv[s]):
        start_diff = float((r["x_ref"].double() - x_maps[c]).abs().max())
        print(f"  {label:<17} chain {c}  map_seed {r['map_seed']}  starts at MAP {c} "
              f"(max diff {start_diff:.1e})  S {r['samples'].shape[0]}  events {r['n_events']:,}  "
              f"grads {r['gradient_evals']:,}  violations {r['bound_violations']}  "
              f"T {r['reservoir']['t_end']:.3g}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 2. Cross-chain comparison on statistics that are invariant to
# permuting hidden units and flipping signs.
#
# The draws of these runs are stored in random order without their times, so
# there is no trace in time and no within-chain ESS. What remains valid is to
# compare the 4 chains' distributions. If a chain still remembers its starting
# MAP, its distribution differs from the others. R-hat (rank-normalized, split)
# is valid here, since splitting shuffled draws just gives two random halves.
# ---------------------------------------------------------------------------
import arviz as az
import matplotlib.pyplot as plt

LOG_2PI = math.log(2 * math.pi)
Xtr, ytr = cdata["X_train"], cdata["y_train"]
Xte, yte = cdata["X_test"], cdata["y_test"]

# Weight-matrix slices in the flat layout [W0, b0, W1, b1, ...] (log sigma last)
W_SLICES, k = [], 0
for n_in, n_out in zip(LAYERS[:-1], LAYERS[1:]):
    W_SLICES.append(slice(k, k + n_in * n_out))
    k += n_in * n_out + n_out


@torch.no_grad()
def invariant_stats(z: Tensor) -> tuple[dict[str, Tensor], Tensor]:
    """Per-draw invariant statistics of z = (beta, log sigma), [S, D] float64.
    Returns ({name: [S]}, test predictions [S, N_test])."""
    beta, log_sigma = z[:, :-1], z[:, -1]
    f_tr = predict(beta, LAYERS, "tanh", Xtr, chunk=256)
    f_te = predict(beta, LAYERS, "tanh", Xte, chunk=256)
    n = ytr.shape[0]
    r2 = ((ytr[None, :] - f_tr) ** 2).sum(1)
    out = {
        "log_sigma":    log_sigma,
        "rmse_train":   (r2 / n).sqrt() * c_ystd,
        "loglik_train": -0.5 * n * LOG_2PI - n * log_sigma - 0.5 * r2 / (2 * log_sigma).exp(),
        "rmse_test":    ((yte[None, :] - f_te) ** 2).mean(1).sqrt() * c_ystd,
    }
    for l, sl in enumerate(W_SLICES):
        out[f"fro_W{l}"] = beta[:, sl].norm(dim=1)
        out[f"zero_W{l}"] = (beta[:, sl] == 0).double().mean(1)
    return out, f_te


# --- statistics for every draw of every chain ----------------------------------
stats, f_tests = {}, {}
for s, label in CONV_SAMPLERS.items():
    per_chain, f_per_chain = [], []
    for c in range(N_CONV_CHAINS):
        z = conv[s][c]["samples"].double()
        o, f_te = invariant_stats(z)
        per_chain.append(o)
        f_per_chain.append(f_te)
        del z
        print(f"  {label} chain {c} done")
    stats[s] = {name: torch.stack([o[name] for o in per_chain]).numpy() for name in per_chain[0]}  # [C, S]
    f_tests[s] = torch.stack(f_per_chain)                                                         # [C, S, N_test]

# the same statistics at the 4 starting MAPs, as reference lines
map_stats, f_maps_te = invariant_stats(x_maps)
STAT_NAMES = list(map_stats)

# --- R-hat per statistic --------------------------------------------------------
print(f"\n{'statistic':<14}" + "".join(f"{l:>20}" for l in CONV_SAMPLERS.values()))
for name in STAT_NAMES:
    cells = ""
    for s in CONV_SAMPLERS:
        r = float(az.rhat(az.convert_to_dataset({name: stats[s][name]}), method="rank")[name])
        cells += f"{r:>20.3f}"
    print(f"{name:<14}{cells}")
for s, label in CONV_SAMPLERS.items():
    ds = az.convert_to_dataset({"f_test": f_tests[s].numpy()})
    r = np.asarray(az.rhat(ds, method="rank")["f_test"])
    print(f"f_test ({label}): R-hat over the {r.size} test points, median {np.median(r):.3f}, max {r.max():.3f}")
print("R-hat below about 1.01 means the chains agree, above 1.05 means they clearly do not.")

# --- agreement of the chains' posterior predictive means ------------------------
# Compare to the MAPs, whose test predictions differed by 0.3 to 1.6 (RMS, y units).
for s, label in CONV_SAMPLERS.items():
    means = f_tests[s].mean(1)                                   # [C, N_test]
    within_sd = float(f_tests[s].std(1).mean()) * c_ystd         # mean posterior sd of f
    print(f"\n{label}, predictive mean per chain (test RMSE) and pairwise RMS differences, y units")
    print("  test RMSE per chain " + "  ".join(f"{conv_rmse(means[c], yte):.3f}" for c in range(N_CONV_CHAINS))
          + f"   (mean posterior sd of f {within_sd:.3f})")
    for i, j in itertools.combinations(range(N_CONV_CHAINS), 2):
        print(f"  chains {i}-{j}  {conv_rmse(means[i], means[j]):.3f}   (their MAPs {conv_rmse(f_maps_te[i], f_maps_te[j]):.3f})")


In [ ]:
import pandas as pd


def rhat(a):
    return float(az.rhat(a, method="rank"))


w_sizes = np.array([sl.stop - sl.start for sl in W_SLICES])
rows = []
for s, label in CONV_SAMPLERS.items():
    st = stats[s]
    zero = sum(st[f"zero_W{l}"] * n for l, n in enumerate(w_sizes)) / w_sizes.sum()   # share of all weights at zero, [C, S]
    rows.append({"network": "large", "sampler": label,
                 "R-hat log sigma": rhat(st["log_sigma"]),
                 "R-hat train RMSE": rhat(st["rmse_train"]),
                 "R-hat test RMSE": rhat(st["rmse_test"]),
                 "R-hat zero %": rhat(100 * zero),
                 "chains": st["log_sigma"].shape[0],
                 "draws per chain": st["log_sigma"].shape[1]})
conv_df = pd.DataFrame(rows).set_index(["network", "sampler"])

rhat_cols = [c for c in conv_df.columns if c.startswith("R-hat")]
conv_df.style.format({c: "{:.3f}" for c in rhat_cols}).background_gradient(
    cmap="Reds", vmin=1.0, vmax=1.5, subset=rhat_cols)


In [ ]:

# --- overlaid histograms per chain, dashed lines at each chain's starting MAP ---
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for s, label in CONV_SAMPLERS.items():
    ncol = 4
    nrow = math.ceil(len(STAT_NAMES) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.8 * nrow))
    for ax, name in zip(axes.ravel(), STAT_NAMES):
        for c in range(N_CONV_CHAINS):
            ax.hist(stats[s][name][c], bins=40, density=True, histtype="step", lw=1.3,
                    color=colors[c], label=f"chain {c}")
            if not name.startswith("zero_"):
                ax.axvline(float(map_stats[name][c]), color=colors[c], ls="--", lw=0.9)
        ax.set_title(name, fontsize=10)
    for ax in axes.ravel()[len(STAT_NAMES):]:
        ax.axis("off")
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle(f"{label}, 4 chains from 4 MAPs (dashed, the chain's starting MAP)")
    plt.tight_layout()
    plt.show()